## Question

What does this notebook investigate, and what would change our mind?


## Why this test exists

The real ACN-Data experiment is the headline test of the paper. Stages 1-8 developed and audited the methodology with synthetic and placeholder data; Stage 9 is the only stage that uses real ACN-Data session records. If F1 / F2 do not generalize to real behavioral uncertainty, the methodological claims are not supported. This notebook runs the full E.2-E.15 protocol end-to-end with the frozen methodology and reports the result.


## Method

The 20-step protocol in `docs/STAGE_10_REAL_DATA_PROTOCOL.md` Appendix E. Each step is a function call to the existing Python modules (`stage5/uncertainty.py`, `stage6/robust_qaoa.py`, `stage9/real_experiment.py`). The driver writes sanitized artifacts to `artifacts/real_*.json` and `artifacts/stage9_*.json` and prints an 8-line summary to stdout. This notebook reads the artifacts after the driver has run and renders them as tables, distributions, and a discussion.


# 10 — Real ACN-Data experiment (PENDING until token in env)

## Credential-safety rules for this notebook

This notebook checks whether `ACN_API_TOKEN` is present in the environment as a **boolean only**. It **never** displays, logs, stores, hashes, or otherwise exposes the token's value, length, prefix, suffix, base64 encoding, or any header that contains it. The notebook's outputs are sanitized; if any token-shaped string appears in an output, the cell is a bug and must be reported.

## Frozen configuration

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.

**No quantum-advantage claim.** This work does NOT claim QAOA outperforms classical optimization. The 11-qubit instance is small enough that exact classical optimization is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. Any quantum-advantage language is explicitly avoided.

## How to run this notebook

From a PowerShell session where `ACN_API_TOKEN` is set in the environment, execute:

```powershell
cd C:\Projects\uncertainity_aware_quantum_opt
python -m stage9.real_experiment
```

The driver writes all artifacts to `artifacts/real_*.json` and `artifacts/stage9_*.json`. The driver prints a sanitized 8-line summary to stdout. Paste the 8-line summary into this notebook for the per-record counters.

## What this notebook does

This notebook follows the 20-step protocol in `docs/STAGE_10_REAL_DATA_PROTOCOL.md` (Appendix E). Concretely:

1. **Credential-availability check** (boolean only).
2. **Acquisition** — `load_real_uncertainty()` fetches from    `https://ev.caltech.edu/api/v1/sessions/caltech`.
3. **Schema validation** — every record's required fields are    checked.
4. **Cleaning** — R9-R11 (per-uncertainty-variable).
5. **Temporal split** — frozen calendar split.
6. **Real ΔE** — sign convention: `E_delivered − E_requested`.
7. **Real Δd** — sign convention: `d_requested − d_actual`.
8. **Empirical distribution** — calibration-only.
9. **Frozen K=8 scenario generation** — seed `20260829 + 8`.
10. **Frozen γ** — pre-registration on calibration only.
11. **F0/F1/F2/F3** — built with frozen penalties.
12. **Exact optimization** — exhaustive on 2^11 bitstrings.
13. **QAOA** — p=1, COBYLA, seeds [0,1,2], shots 1024.
14. **Held-out replay** — per-session per-formulation.
15. **P(feasible)** — primary outcome.
16. **Unmet demand** — per session, per formulation.
17. **Cost** — per session, per formulation.
18. **Peak / grid impact** — per session, per formulation.
19. **Paired statistical comparisons** — bootstrap CI, Bonferroni.
20. **Interpretation** — Case A / B / C applied honestly.

## What this notebook does NOT do

- It does not modify the frozen methodology.
- It does not re-tune γ, K, α, M_window, ρ_d, ρ_p, ρ_cap, or any   QAOA setting based on held-out data.
- It does not claim quantum advantage.
- It does not synthesize or substitute data for the real data.
- It does not display the token or any credential.


## Implementation


In [ ]:
import os
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))

# Step 1: credential-availability check (boolean only)
_token_present = bool(os.environ.get('ACN_API_TOKEN')
                      or os.environ.get('ACNPORTAL_TOKEN'))
print(f"ACN_API_TOKEN or ACNPORTAL_TOKEN present: {_token_present}")
if not _token_present:
    print("Set the env var in this PowerShell session and re-run.")
    print("Do NOT paste the token into a notebook cell.")
else:
    print("Token is present. Do not display, log, or print its value here.")
    print("The loader (stage5/uncertainty.py::load_real_uncertainty) will use it directly.")


## Real-data acquisition (steps 2-5)

Once the token is set, run the Stage 9 driver from this same PowerShell session. The driver performs acquisition, schema validation, cleaning, and the temporal split, and writes `artifacts/real_raw_sessions.json`, `artifacts/real_cleaning_results.json`, `artifacts/real_uncertainty_statistics.json`.

**No token-bearing material is in any of these files.** The loader never logs the token; the `_redact_status` defensive filter scans the status block for token-shaped strings and redacts them before write.


In [ ]:
import pathlib, json
real_artifacts = sorted(pathlib.Path('../artifacts').glob('real_*.json'))
for p in real_artifacts:
    info = json.loads(p.read_text(encoding='utf-8'))
    print(f"{p.name}:")
    if isinstance(info, dict):
        for k, v in info.items():
            if k.startswith('_'): continue
            if isinstance(v, (int, float, str, bool)):
                print(f"  {k}: {v}")
    print()
if not real_artifacts:
    print('No real_* artifacts yet. Run the driver (Step 2) to produce them.')


## Real ΔE and Δd distributions (steps 6-8)

After the driver runs, `artifacts/real_uncertainty_statistics.json` contains:

- `calibration.Delta_d_distribution` (mean, median, std, 1/5/10/25/  50/75/90/95/99 quantiles)
- `calibration.Delta_e_distribution` (same)
- `calibration.Delta_d_sign_breakdown` (positive / zero / negative)
- `calibration.Delta_e_sign_breakdown` (same)
- `calibration.joint_dependence` (Pearson / Spearman)

**Interpretation rule:** if the real ΔE / Δd distributions differ materially from the placeholder, the discussion must explain the implications for F1/F2 (which are calibrated to the empirical distribution). No re-tuning is performed.


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_uncertainty_statistics.json')
if p.exists():
    s = json.loads(p.read_text())
    if 'calibration' in s:
        print('CALIBRATION Δd:', s['calibration'].get('Delta_d_distribution'))
        print('CALIBRATION ΔE:', s['calibration'].get('Delta_e_distribution'))
        print('CALIBRATION joint:', s['calibration'].get('joint_dependence'))
    if 'held_out' in s:
        print()
        print('HELD-OUT Δd:', s['held_out'].get('Delta_d_distribution'))
        print('HELD-OUT ΔE:', s['held_out'].get('Delta_e_distribution'))
    print()
    print(f"missing-data (calibration): {s.get('missing_data_report_calibration', {})}")
else:
    print('real_uncertainty_statistics.json not produced yet.')


## Frozen K=8 and γ (steps 9-10)

After the driver runs, `artifacts/real_scenarios_K8.json` contains the 8 cluster centroids and weights from the calibration joint. `artifacts/real_calibration_parameters.json` contains γ, frozen before any held-out use.


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_calibration_parameters.json')
if p.exists():
    s = json.loads(p.read_text())
    print(f"real γ (frozen on calibration): {s.get('gamma')}")
    print(f"real ρ_d_robust: {s.get('rho_d_robust')}")
    print(f"frozen at: {s.get('frozen_at_utc')}")
else:
    print('real_calibration_parameters.json not produced yet.')


## F0/F1/F2/F3 + exact + QAOA + held-out + statistics (steps 11-19)

All of these are produced by the Stage 9 driver and written to `artifacts/real_f0_results.json`, `real_f1_robust_results.json`, `real_f2_adopt_results.json`, `real_f3_oracle_results.json`, `real_heldout_results.json`, `real_paired_statistics.json`, `real_distribution_shift.json`.


In [ ]:
import json, pathlib
for name in ('real_f0_results','real_f1_robust_results','real_f2_adopt_results','real_f3_oracle_results'):
    p = pathlib.Path(f'../artifacts/{name}.json')
    if p.exists():
        s = json.loads(p.read_text())
        print(f'{name}: classical_optimum = {s.get("classical_optimum")}')
    else:
        print(f'{name}: not produced yet.')


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_heldout_results.json')
if p.exists():
    s = json.loads(p.read_text())
    for fname, h in s.items():
        print(f"{fname}: P(feas) = {h.get('P_feasible')}  mean_unmet = {h.get('mean_unmet_kWh')}  deadline_viol = {h.get('deadline_violation_rate')}")
else:
    print('real_heldout_results.json not produced yet.')


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_paired_statistics.json')
if p.exists():
    s = json.loads(p.read_text())
    prim = s.get('primary_F2_vs_F0', {})
    print(f"Primary F2 vs F0 P(feasible) diff = {prim.get('diff_mean')}")
    print(f"95% CI = [{prim.get('ci_lo')}, {prim.get('ci_hi')}]")
    print(f"Bonferroni α = {s.get('bonferroni_alpha')}")
else:
    print('real_paired_statistics.json not produced yet.')


## Interpretation (step 20)

Apply the Case A / B / C framework to the distribution-shift diagnostic:

- **Case A (mild shift)**: Cohen's d < 0.2 on both marginals. F0/F1/F2   ranking is expected to be similar on calibration and held-out.
- **Case B (moderate shift)**: Cohen's d in [0.2, 0.5). F1/F2 may   differ from F0 in ranking; the bootstrap CI on F2 vs F0 is the   primary endpoint.
- **Case C (severe shift)**: Cohen's d ≥ 0.5. The frozen γ may not   be calibrated for the held-out distribution; the paper reports   the result **as-is**, with the discussion noting the limitation.

**No re-tuning is performed in any case.** The frozen methodology remains frozen. The result is reported honestly, favorable or not.


In [ ]:
import json, pathlib
p = pathlib.Path('../artifacts/real_distribution_shift.json')
if p.exists():
    s = json.loads(p.read_text())
    print(f"case_label: {s.get('case_label')}")
    for k in ('Delta_d_cal_mean', 'Delta_d_ho_mean', 'Delta_e_cal_mean', 'Delta_e_ho_mean'):
        print(f"  {k}: {s.get(k)}")
else:
    print('real_distribution_shift.json not produced yet.')


## Limitations

- The 11-qubit instance is small; the result does not predict   behavior on larger instances.
- γ is a single scalar; it cannot represent full distributional   change.
- The held-out window is 6 months; longer windows would give more   robust deployment estimates.
- The Caltech site is one of several ACN-Data sites; the result   may not generalize to JPL, Caltech Office, etc.
- The frozen QAOA configuration (p=1, COBYLA, seeds [0,1,2], shots   1024) is the only one reported; p=2 and other configurations are   sensitivity checks, not headline results.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Result

See the code outputs above for the numerical results. The interpretation is in the next section.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.
